# CIFAR-10 Image Classification with a CNN

This notebook trains a simple convolutional neural network on the CIFAR-10 dataset (60,000 small 32x32 colour images in 10 classes such as airplane, cat and truck). It evaluates the model with test accuracy and a confusion matrix, saves it, and then uses it to predict the class of a new image.

**How to run:** Open in Google Colab, choose *Runtime → Change runtime type → GPU*, then run all cells. The dataset downloads automatically through `tf.keras.datasets`.

_Rewritten from my original Colab notebook: `CIFAR.ipynb`._

## 1. Imports and seeds

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

class_names = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

## 2. Load and prepare the data

Pixel values are scaled to 0–1. Labels are kept as integers, so we use `sparse_categorical_crossentropy`.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
y_train = y_train.flatten()
y_test = y_test.flatten()
print("Train:", x_train.shape, "Test:", x_test.shape)

plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i]])
    plt.axis("off")
plt.show()

## 3. Build the model

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu", padding="same"),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 4. Train

In [ ]:
early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
history = model.fit(x_train, y_train, epochs=20, batch_size=128,
                    validation_split=0.2, callbacks=[early_stop])

plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="validation")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend(); plt.show()

## 5. Evaluate

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

y_pred = np.argmax(model.predict(x_test), axis=1)
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", xticks_rotation=45)
plt.title("Confusion matrix (test set)")
plt.show()

## 6. Save and reload the model

In [ ]:
model.save("cifar10_cnn.keras")
loaded_model = keras.models.load_model("cifar10_cnn.keras")

## 7. Predict a new image

Any image is converted to RGB and resized to 32x32 (so PNGs with transparency or grayscale images also work). Here we save a test image to a file and load it back; replace `image_path` with your own uploaded file to try it.

In [ ]:
from PIL import Image

def predict_image(image):
    image = image.convert("RGB").resize((32, 32))
    arr = np.array(image, dtype="float32") / 255.0
    probs = loaded_model.predict(np.expand_dims(arr, axis=0), verbose=0)[0]
    best = int(np.argmax(probs))
    return class_names[best], float(probs[best])

Image.fromarray((x_test[0] * 255).astype("uint8")).save("example.png")
image_path = "example.png"  # change to your own image file
img = Image.open(image_path)
label, conf = predict_image(img)
plt.imshow(img); plt.axis("off"); plt.title(f"Predicted: {label} ({conf:.2f})"); plt.show()
print("True label:", class_names[y_test[0]])